In [44]:
import sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [45]:
db_path = Path("../../data/database.sqlite")
conn = sqlite3.connect(db_path.as_posix())
df_agg = pd.read_sql_query("SELECT * FROM Match_Aggregated", conn)
df_match = pd.read_sql_query("SELECT * FROM Match_Cleaned", conn)
conn.close()

In [46]:
df_agg.info()

<class 'pandas.DataFrame'>
RangeIndex: 427220 entries, 0 to 427219
Data columns (total 84 columns):
 #   Column                                    Non-Null Count   Dtype  
---  ------                                    --------------   -----  
 0   match_api_id                              427220 non-null  int64  
 1   date                                      427220 non-null  str    
 2   position_col                              427220 non-null  str    
 3   player_api_id                             427220 non-null  int64  
 4   team_side                                 427220 non-null  str    
 5   position_no                               427220 non-null  int64  
 6   id                                        427220 non-null  int64  
 7   player_fifa_api_id                        427220 non-null  int64  
 8   overall_rating                            427220 non-null  float64
 9   potential                                 427220 non-null  float64
 10  preferred_foot                 

In [47]:
df_match_fc_bayern_munich = df_agg[
    (df_agg["match_api_id"] == 2002277) & (df_agg["team_side"] == "away")
]
df_match_fc_bayern_munich.head()

,match_api_id,date,position_col,player_api_id,team_side,position_no,id,player_fifa_api_id,overall_rating,potential,...,away_team_chanceCreationShooting,away_team_chanceCreationPositioningClass,away_team_defencePressure,away_team_defenceAggression,away_team_defenceTeamWidth,away_team_defenceDefenderLineClass,away_team_team_long_name,away_team_team_short_name,player_x,player_y
406790,2002277,2016-02-14 00:00:00,away_player_7,30834,away,7,16461,9014,89.0,89.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,2.0,8.0
406794,2002277,2016-02-14 00:00:00,away_player_2,30894,away,2,143229,121939,87.0,87.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,2.0,3.0
406890,2002277,2016-02-14 00:00:00,away_player_6,49939,away,6,17411,181872,85.0,85.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,5.0,6.0
406927,2002277,2016-02-14 00:00:00,away_player_11,93447,away,11,150542,188545,88.0,89.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,5.0,11.0
406961,2002277,2016-02-14 00:00:00,away_player_8,116772,away,8,170800,189596,86.0,88.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,4.0,8.0


# Model 1

In [48]:
class FormationTower(nn.Module):
    def __init__(self):
        super().__init__()
        # Phi: (B, 9, 2) -> (B, 9, 64)
        self.phi = nn.Sequential(
            nn.Linear(in_features=2, out_features=64),
            nn.LeakyReLU(),
            nn.Linear(in_features=64, out_features=64),
            nn.LeakyReLU(),
            nn.Linear(in_features=64, out_features=64),
            nn.LeakyReLU(),
        )

        # Rho: (B, 64 + 2) -> (B, 64)
        # We match in_features to the phi output (64) + position (2)
        self.rho = nn.Sequential(
            nn.Linear(in_features=64 + 2, out_features=128),
            nn.BatchNorm1d(128),  # Must match out_features above
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),  # Final embedding dim: 64
        )

    def forward(self, formation, position):
        x = self.phi(formation)  # (Batch, 9, 64)
        x = torch.sum(x, dim=1)  # (Batch, 64)

        x = torch.concat([x, position], dim=1)  # (Batch, 66)
        return self.rho(x)


class PlayerTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features=31, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),  # Final embedding dim: 64
        )

    def forward(self, player):
        return self.mlp(player)

In [49]:
class TwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.formation_tower = FormationTower()
        self.player_tower = PlayerTower()

    def forward(self, formation, position, player):
        formation_embedding = self.formation_tower(formation, position)
        player_embedding = self.player_tower(player)

        f_norm = formation_embedding / (
            torch.linalg.norm(formation_embedding, dim=1, keepdim=True) + 1e-8
        )
        p_norm = player_embedding / (
            torch.linalg.norm(player_embedding, dim=1, keepdim=True) + 1e-8
        )

        similarity_score = (f_norm * p_norm).sum(dim=1)

        return f_norm, p_norm, similarity_score

In [50]:
player_stats_cols = [
    "preferred_foot",
    "attacking_work_rate",
    "defensive_work_rate",
    "crossing",
    "finishing",
    "heading_accuracy",
    "short_passing",
    "volleys",
    "dribbling",
    "curve",
    "free_kick_accuracy",
    "long_passing",
    "ball_control",
    "acceleration",
    "sprint_speed",
    "agility",
    "reactions",
    "balance",
    "shot_power",
    "jumping",
    "stamina",
    "strength",
    "long_shots",
    "aggression",
    "interceptions",
    "positioning",
    "vision",
    "penalties",
    "marking",
    "standing_tackle",
    "sliding_tackle",
]

In [51]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def rate_bayern_squad_for_robben_spot(target_name, match_df, model, device):
    model.eval()

    # 1. Extract the "Fixed" Tactical Context of Arjen Robben
    # We define the "Job" based on Robben's coordinates in this specific match
    robben_row = match_df[match_df["player_name"] == target_name].iloc[0]
    teammates_df = match_df[match_df["player_name"] != target_name]

    # Global normalization for Robben's spot
    target_x_norm = (robben_row["player_x"] - 5.0) / 4.0
    target_y_norm = (robben_row["player_y"] - 7.0) / 4.0
    target_pos = torch.tensor([[target_x_norm, target_y_norm]], dtype=torch.float32).to(
        device
    )

    # Relative geometry for the 9 teammates around that spot
    tm_pos = teammates_df[["player_x", "player_y"]].to_numpy().astype(float)
    tm_pos_norm = tm_pos.copy()
    tm_pos_norm[:, 0] = (tm_pos_norm[:, 0] - 5.0) / 4.0
    tm_pos_norm[:, 1] = (tm_pos_norm[:, 1] - 7.0) / 4.0

    relative_vectors = tm_pos_norm - np.array([target_x_norm, target_y_norm])
    distances = np.linalg.norm(relative_vectors, axis=1) / 2.5
    angles = np.arctan2(relative_vectors[:, 1], relative_vectors[:, 0]) / np.pi

    formation_input = np.stack([distances, angles], axis=1)[:9]
    formation_tensor = torch.tensor([formation_input], dtype=torch.float32).to(device)

    # 2. Score Every Player in the Squad for this Spot
    results = []

    with torch.no_grad():
        # Encode the "Job Description" (The Fixed Formation)
        f_embed = model.formation_tower(formation_tensor, target_pos)
        f_norm = f_embed / (torch.linalg.norm(f_embed, dim=1, keepdim=True) + 1e-8)

        for _, player_row in match_df.iterrows():
            # Get this player's stats
            p_stats = torch.tensor(
                [player_row[player_stats_cols].values], dtype=torch.float32
            ).to(device)

            # Encode the Player
            p_embed = model.player_tower(p_stats)
            p_norm = p_embed / (torch.linalg.norm(p_embed, dim=1, keepdim=True) + 1e-8)

            # Calculate Similarity Score (0 to 1)
            # Dot product of normalized vectors = Cosine Similarity
            score = torch.matmul(f_norm, p_norm.T).item()

            results.append(
                {
                    "Name": player_row["player_name"],
                    "Tactical_Fit_Score": round(score, 4),
                    "Original_Pos": player_row["position_col"],  # For reference
                }
            )

    # Sort results by score
    df_scores = pd.DataFrame(results).sort_values(
        by="Tactical_Fit_Score", ascending=False
    )
    return df_scores


# --- RUN THE TEST ---
model = TwoTower().to(device)
model_path = "best_soccer_model.pth"
state_dict = torch.load(model_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)

squad_rankings = rate_bayern_squad_for_robben_spot(
    "Robert Lewandowski", df_match_fc_bayern_munich, model, device
)

print("How the Bayern Squad fits into Arjen Robben's Tactical Spot:")
print(squad_rankings.to_string(index=False))

How the Bayern Squad fits into Arjen Robben's Tactical Spot:
              Name  Tactical_Fit_Score   Original_Pos
Robert Lewandowski              0.0601 away_player_11
      Arjen Robben              0.0352  away_player_7
  Thiago Alcantara              0.0018  away_player_9
     Douglas Costa             -0.0101 away_player_10
    Thomas Mueller             -0.0285  away_player_8
       Juan Bernat             -0.0568  away_player_5
      Arturo Vidal             -0.0598  away_player_6
    Joshua Kimmich             -0.0603  away_player_3
      Philipp Lahm             -0.0636  away_player_2
       David Alaba             -0.0933  away_player_4


# Model 2

In [63]:
class PositionTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features=2, out_features=64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(),
            nn.Linear(in_features=64, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),
        )

    def forward(self, x):
        return self.net(x)


class PlayerTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features=31, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),
        )

    def forward(self, x):
        return self.net(x)


class SimpleTwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.position_tower = PositionTower()
        self.player_tower = PlayerTower()

    def forward(self, position, player):
        pos_emb = self.position_tower(position)
        plr_emb = self.player_tower(player)

        p_norm = pos_emb / (torch.linalg.norm(pos_emb, dim=1, keepdim=True) + 1e-8)
        u_norm = plr_emb / (torch.linalg.norm(plr_emb, dim=1, keepdim=True) + 1e-8)
        return p_norm, u_norm


# --- 2. THE SIMPLIFIED RATING FUNCTION ---


def rate_bayern_squad_simple(target_name, match_df, model, device):
    model.eval()

    # 1. Extract the Fixed (x, y) of the target spot (e.g., Robben's RW spot)
    target_row = match_df[match_df["player_name"] == target_name].iloc[0]

    # Normalize (x, y) exactly like training
    target_x_norm = (target_row["player_x"] - 5.0) / 4.0
    target_y_norm = (target_row["player_y"] - 7.0) / 4.0
    target_pos = torch.tensor([[target_x_norm, target_y_norm]], dtype=torch.float32).to(
        device
    )

    results = []

    with torch.no_grad():
        # Encode the 'Position Requirement'
        # Simple model: position_tower only needs the (x, y)
        p_norm = model.position_tower(target_pos)
        p_norm = p_norm / (torch.linalg.norm(p_norm, dim=1, keepdim=True) + 1e-8)

        for _, player_row in match_df.iterrows():
            # Get player stats
            p_stats = torch.tensor(
                [player_row[player_stats_cols].values], dtype=torch.float32
            ).to(device)

            # Encode Player
            u_norm = model.player_tower(p_stats)
            u_norm = u_norm / (torch.linalg.norm(u_norm, dim=1, keepdim=True) + 1e-8)

            # Score
            score = torch.matmul(p_norm, u_norm.T).item()

            results.append(
                {
                    "Name": player_row["player_name"],
                    "Tactical_Fit_Score": round(score, 4),
                    "Original_Pos": player_row["position_col"],
                }
            )

    df_scores = pd.DataFrame(results).sort_values(
        by="Tactical_Fit_Score", ascending=False
    )
    return df_scores


# --- 3. RUN THE TEST ---

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the Simple Model
model = SimpleTwoTower().to(device)
model_path = "simple_soccer_model.pth"  # Ensure you saved the simple model here
state_dict = torch.load(model_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)

# Calculate Rankings for Robben's spot
squad_rankings = rate_bayern_squad_simple(
    "Arjen Robben", df_match_fc_bayern_munich, model, device
)

print("Results for Simple Model (Position Only)")
print(squad_rankings.to_string(index=False))

Results for Simple Model (Position Only)
              Name  Tactical_Fit_Score   Original_Pos
       David Alaba             -0.0412  away_player_4
      Arjen Robben             -0.0457  away_player_7
    Thomas Mueller             -0.0497  away_player_8
     Douglas Costa             -0.0538 away_player_10
       Juan Bernat             -0.0638  away_player_5
      Arturo Vidal             -0.0666  away_player_6
      Philipp Lahm             -0.0977  away_player_2
Robert Lewandowski             -0.1120 away_player_11
    Joshua Kimmich             -0.1450  away_player_3
  Thiago Alcantara             -0.1958  away_player_9
